# Chapter 6: Finetuning for Sociolinguistics Classification Bahasa Indonesia (SCBI)

Adapted from *Build a Large Language Model From Scratch* by Sebastian Raschka.
Instead of binary spam classification, we finetune GPT-2 to classify Indonesian text into **3 sociolinguistic categories**:

| Label | Class | Description |
|-------|-------|-------------|
| 0 | **Alay** | Indonesian slang / informal abbreviated text |
| 1 | **EYD** | Formal Indonesian (standard spelling rules) |
| 2 | **Jaksel** | Jakarta Selatan style (Indonesian mixed with English) |

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib", "numpy", "tiktoken", "torch", "tensorflow", "pandas"]
for p in pkgs:
    print(f"{p} version: {version(p)}")

## 6.1 Different categories of finetuning

- No code in this section
- We perform **classification finetuning** for our 3-class (Alay, EYD, Jaksel) task

## 6.2 Preparing the dataset

- We load cleaned datasets from `cleaned-data/` and combine them

In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path("cleaned-data")

alay_df = pd.read_csv(data_dir / "alay_cleaned.csv")
eyd_df = pd.read_csv(data_dir / "eyd_cleaned.csv")
jaksel_df = pd.read_csv(data_dir / "jaksel_cleaned.csv")

print(f"Alay samples:   {len(alay_df)}")
print(f"EYD samples:    {len(eyd_df)}")
print(f"Jaksel samples: {len(jaksel_df)}")

for d in [alay_df, eyd_df, jaksel_df]:
    d.rename(columns={"text": "Text", "label": "Label"}, inplace=True)

df = pd.concat([alay_df, eyd_df, jaksel_df], ignore_index=True)
df = df.dropna(subset=["Text"]).reset_index(drop=True)
print(f"\nTotal combined: {len(df)}")
df

- Class distribution:

In [ ]:
print(df["Label"].value_counts())

- Undersample to balance all three classes

In [ ]:
def create_balanced_dataset(df):
    min_count = df["Label"].value_counts().min()
    balanced_dfs = []
    for label in df["Label"].unique():
        subset = df[df["Label"] == label].sample(min_count, random_state=123)
        balanced_dfs.append(subset)
    return pd.concat(balanced_dfs)

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

- Map string labels to integers: alay=0, eyd=1, jaksel=2

In [ ]:
balanced_df["Label"] = balanced_df["Label"].map({"alay": 0, "eyd": 1, "jaksel": 2})
print(balanced_df["Label"].value_counts())

- Split into train (70%), validation (10%), test (20%)

In [ ]:
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = int(len(df) * train_frac)
    val_end = train_end + int(len(df) * validation_frac)
    return df[:train_end], df[train_end:val_end], df[val_end:]

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

print(f"Train: {len(train_df)}, Val: {len(validation_df)}, Test: {len(test_df)}")

## 6.3 Creating data loaders

- We pad all messages to the longest sequence length using `<|endoftext|>` as a padding token (token ID 50256)

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

In [ ]:
import torch
from torch.utils.data import Dataset


class SCBIDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # Pre-tokenize texts
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

In [ ]:
train_dataset = SCBIDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer
)
print(train_dataset.max_length)

- Pad validation and test sets to match training set max length

In [ ]:
val_dataset = SCBIDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SCBIDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

- Create data loaders

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

- Verify batch shapes

In [ ]:
print("Train loader:")
for input_batch, target_batch in train_loader:
    pass
print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

In [ ]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

## 6.4 Initializing a model with pretrained weights

- First, we download the helper modules from the LLMs-from-scratch repository

In [ ]:
import urllib.request
import os

url_base = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch06/01_main-chapter-code/"

files_to_download = ["previous_chapters.py", "gpt_download.py"]

for f in files_to_download:
    if not os.path.exists(f):
        url = url_base + f
        print(f"Downloading {f}...")
        urllib.request.urlretrieve(url, f)
        print(f"  Saved {f}")
    else:
        print(f"{f} already exists.")

- Initialize the GPT-2 model with pretrained weights

In [ ]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"Dataset length {train_dataset.max_length} exceeds model's context "
    f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with "
    f"`max_length={BASE_CONFIG['context_length']}`"
)

In [ ]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

- Verify the model generates coherent text

In [ ]:
from previous_chapters import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)

text_1 = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

## 6.5 Adding a classification head

- We modify the pretrained LLM for classification finetuning
- First, freeze all layers

In [ ]:
print(model)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

- Replace the output layer for **3-class** classification (Alay, EYD, Jaksel)

In [ ]:
torch.manual_seed(123)

num_classes = 3
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

- Also make the last transformer block and final LayerNorm trainable

In [ ]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

- Feed some text input to verify the model now outputs 3 dimensions

In [ ]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape)

In [ ]:
with torch.no_grad():
    outputs = model(inputs)

print("Outputs:\n", outputs)
print("Outputs dimensions:", outputs.shape) # (batch_size, num_tokens, num_classes=3)

- The last token contains the most information due to causal attention

In [ ]:
print("Last output token:", outputs[:, -1, :])

## 6.6 Calculating the classification loss and accuracy

In [ ]:
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print("Class label:", label.item())

- The `calc_accuracy_loader` computes the percentage of correct predictions

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

- Calculate initial accuracies (before training):

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

torch.manual_seed(123)

train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

- Define the loss functions (cross-entropy works for multi-class)

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

- Compute initial losses before training

In [ ]:
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

## 6.7 Finetuning the model on supervised data

- The `train_classifier_simple` function is the same as in chapter 5, adapted for classification

In [ ]:
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

- Train the model

In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

- Plot training and validation loss

In [ ]:
import matplotlib.pyplot as plt

def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))
    ax1.plot(epochs_seen, train_values, label=f"Training {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"Validation {label}")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()
    ax2 = ax1.twiny()
    ax2.plot(examples_seen, train_values, alpha=0)
    ax2.set_xlabel("Examples seen")
    fig.tight_layout()
    plt.savefig(f"{label}-plot.pdf")
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))
plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

- Plot accuracy

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))
plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

- Compute full dataset accuracies

In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

## 6.8 Using the LLM as a sociolinguistics classifier

- The `classify_text` function preprocesses text and returns the predicted class label

In [ ]:
def classify_text(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[1]
    input_ids = input_ids[:min(max_length, supported_context_length)]
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]
    predicted_label = torch.argmax(logits, dim=-1).item()
    label_map = {0: "Alay", 1: "EYD", 2: "Jaksel"}
    return label_map[predicted_label]

- Let's try it on some example texts:

In [ ]:
# Alay example
text_1 = "gw bgt lah, emg gk bsa tidur smpe pagi"
print(f"Text: {text_1}")
print(f"Predicted: {classify_text(text_1, model, tokenizer, device, max_length=train_dataset.max_length)}")
print()

# EYD example
text_2 = "Pada pagi hari, ibu pergi ke pasar untuk membeli sayuran segar"
print(f"Text: {text_2}")
print(f"Predicted: {classify_text(text_2, model, tokenizer, device, max_length=train_dataset.max_length)}")
print()

# Jaksel example
text_3 = "Literally gue tuh lagi overthinking banget deh about this whole situation"
print(f"Text: {text_3}")
print(f"Predicted: {classify_text(text_3, model, tokenizer, device, max_length=train_dataset.max_length)}")

- Save the finetuned model

In [ ]:
torch.save(model.state_dict(), "scbi_classifier.pth")
print("Model saved to scbi_classifier.pth")

- To load the model in a new session:

In [ ]:
model_state_dict = torch.load("scbi_classifier.pth", map_location=device, weights_only=True)
model.load_state_dict(model_state_dict)

## Summary

- We adapted the GPT-2 classification finetuning pipeline from Chapter 6 for a **3-class** sociolinguistics task
- The model classifies Indonesian text as **Alay** (slang), **EYD** (formal), or **Jaksel** (Indonglish)
- Training used balanced undersampled data from cleaned CSV files
- The same cross-entropy loss and training loop from Chapter 6 work naturally for multi-class classification